<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES.
All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# ViDoRe v3 HR: ingestion and retrieval evaluation

A compact companion to `nrl_ragas.ipynb`: ingest the HR corpus into LanceDB,
retrieve relevant pages, and report **Recall and nDCG at 1, 3, 5, and 10**.
Each stage below explains what it does. There is no answer generation or judge.

Use a **Python 3.12** kernel. Edit the configuration cell, then choose **Run All Cells**.
On first use, enable dependency installation; restart the kernel afterward and disable
installation for subsequent runs. Local mode requires NVIDIA GPUs and sufficient memory
for extraction and embedding models. Hosted mode uses remote HTTP models and needs no GPU.


## 1. Configure the run

`MAX_QUERIES=None` evaluates the complete test split; set a small number for a smoke run.
The **entire corpus** is always ingested so limiting queries does not remove distractors.
`INFERENCE_MODE` selects local extraction/embedding or hosted Page Elements, OCR, and
embedding. Hosted URLs can also point to your own compatible NIM deployments.
Set `NVIDIA_API_KEY` in the kernel environment for authenticated endpoints and `HF_TOKEN`
if Hugging Face access requires authentication (cached Hugging Face login also works).

Indexes are separated by mode and model configuration. Set `OVERWRITE_INDEX=True`
to rebuild. Paths are resolved from the repository root, whether the kernel starts
in the root or in `examples`.


In [ ]:
from pathlib import Path

INFERENCE_MODE = "local"  # "local" or "hosted"
INSTALL_DEPENDENCIES = False  # True on first use; restart kernel afterward.
MAX_QUERIES = None  # e.g. 20 for a smoke run; None for the full test split.
OVERWRITE_INDEX = False
DATASET_REVISION = "main"  # Or a Hugging Face commit SHA for a repeatable run.

LOCAL_EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16"
HOSTED_EMBED_MODEL = "nvidia/nemotron-3-embed-1b"
PAGE_ELEMENTS_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3"
OCR_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-ocr-v2"
EMBED_URL = "https://integrate.api.nvidia.com/v1/embeddings"

QUERY_BATCH_SIZE = 32
INITIAL_CANDIDATES = 40  # Multiple extracted elements may belong to the same page.
MAX_CANDIDATES = 1000  # Increase if retrieval cannot produce 10 unique pages.

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "nemo_retriever" / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "nemo_retriever" / "pyproject.toml").is_file(), "Start in the repository root or examples."
DATA_DIR = REPO_ROOT / "data" / "vidore_v3_hr_notebook"
INDEX_DIR = REPO_ROOT / "examples" / "lancedb-vidore-v3-hr"
RESULTS_DIR = REPO_ROOT / "examples" / "vidore-v3-hr-results"

assert INFERENCE_MODE in {"local", "hosted"}
assert MAX_QUERIES is None or (isinstance(MAX_QUERIES, int) and MAX_QUERIES > 0)
assert QUERY_BATCH_SIZE > 0 and 10 <= INITIAL_CANDIDATES <= MAX_CANDIDATES
EMBED_MODEL = LOCAL_EMBED_MODEL if INFERENCE_MODE == "local" else HOSTED_EMBED_MODEL
CUTOFFS = (1, 3, 5, 10)


## 2. Prepare dependencies and model access

The local extra installs the GPU model stack; hosted mode installs the base package.
Installation is opt-in so Run All does not upgrade a working environment.
Credentials are read from environment variables and never written into notebook outputs.
Hosted mode sends corpus pages/crops and query text to the configured endpoints.


In [ ]:
import os
import shutil
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    target = str(REPO_ROOT / "nemo_retriever")
    if INFERENCE_MODE == "local":
        target += "[local]"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", target,
         "datasets>=4.8.2", "huggingface-hub>=1.5,<2"],
        check=True,
    )
    raise RuntimeError("Installation finished. Restart the kernel, set INSTALL_DEPENDENCIES=False, and Run All.")

import lancedb
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import display
from tqdm.auto import tqdm
from nemo_retriever.graph.retriever import Retriever
from nemo_retriever.tools.recall.beir import (
    build_queries_by_id, build_qrels_by_query_id,
    build_beir_run_from_hits, compute_beir_metrics,
)

os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
retriever_cli = shutil.which("retriever")
assert retriever_cli, "Install nemo-retriever in this kernel environment first."
if INFERENCE_MODE == "local":
    import torch
    assert torch.cuda.is_available(), "Local inference requires CUDA-enabled PyTorch and a visible NVIDIA GPU."
    print(f"Local inference: {torch.cuda.device_count()} GPU(s), embedding model {EMBED_MODEL}")
else:
    print(f"Hosted inference: embedding model {EMBED_MODEL}")


## 3. Load the HR test split and relevance labels

Load the `corpus`, `queries`, and `qrels` partitions from
[ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr).
Resolve the dataset revision to one commit before loading all three partitions.
Qrels associate query IDs with **corpus page IDs**, including relevance scores.
Only queries with positive relevance labels are evaluated; this notebook does not filter by language.


In [ ]:
import hashlib
import json

DATASET_ID = "vidore/vidore_v3_hr"
dataset_sha = HfApi().dataset_info(DATASET_ID, revision=DATASET_REVISION).sha
partitions = {
    name: load_dataset(DATASET_ID, data_dir=name, split="test", revision=dataset_sha)
    for name in ("corpus", "queries", "qrels")
}
corpus = partitions["corpus"]
corpus_ids = {str(value) for value in corpus["corpus_id"]}
assert len(corpus_ids) == len(corpus), "Corpus IDs must be unique."
assert all(value.isdecimal() for value in corpus_ids), "Expected numeric ViDoRe corpus IDs."
query_ids, queries = build_queries_by_id(partitions["queries"])
qrels = build_qrels_by_query_id(partitions["qrels"], allowed_query_ids=set(query_ids))
unknown_ids = {doc_id for gold in qrels.values() for doc_id in gold} - corpus_ids
assert not unknown_ids, f"Qrels reference unknown corpus IDs: {sorted(unknown_ids)[:5]}"
selected = [(qid, query) for qid, query in zip(query_ids, queries)
            if any(score > 0 for score in qrels.get(qid, {}).values())]
if MAX_QUERIES is not None:
    selected = selected[:MAX_QUERIES]
assert selected, "No queries with positive relevance labels found."
query_ids, queries = map(list, zip(*selected))
assert len(set(query_ids)) == len(query_ids), "Query IDs must be unique."
qrels = {qid: qrels[qid] for qid in query_ids}
print(f"Dataset revision: {dataset_sha}")
print(f"Corpus pages: {len(corpus)}; queries to evaluate: {len(queries)}")
print(f"Example query: {queries[0]}")


## 4. Prepare the ingestion corpus

Save each corpus image as a single-page PDF named `<corpus_id>.pdf`. This preserves
an exact mapping from retrieved source filenames to qrels. The image is the ingestion
input; the dataset's reference markdown is not used. Exported PDFs are cached by
dataset revision, and only missing pages are written on subsequent runs.


In [ ]:
corpus_dir = DATA_DIR / dataset_sha / "pdfs"
corpus_dir.mkdir(parents=True, exist_ok=True)
missing_pages = [i for i, value in enumerate(corpus["corpus_id"])
                 if not (corpus_dir / f"{value}.pdf").is_file()]
for row in tqdm(corpus.select(missing_pages), total=len(missing_pages), desc="Exporting corpus pages"):
    image = row["image"].convert("RGB")
    path = corpus_dir / f"{row['corpus_id']}.pdf"
    temporary = path.with_suffix(".tmp")
    image.save(temporary, format="PDF", resolution=150.0)
    temporary.replace(path)
assert {path.stem for path in corpus_dir.glob("*.pdf")} == corpus_ids
print(f"Ready to ingest {len(corpus_ids)} single-page PDFs from {corpus_dir}")


## 5. Ingest or reuse the LanceDB index

NeMo Retriever extracts page content and embeds it into LanceDB. Local mode uses
local extraction models and the Hugging Face embedding backend. Hosted mode passes
the configured HTTP endpoints to ingestion and uses the same embedding endpoint for queries.
Ingestion streams CLI progress and can take a while for the full corpus.

A configuration manifest protects index reuse: an incomplete ingestion or incompatible
index requires a rebuild. Model or endpoint changes create a separate index directory.


In [ ]:
index_config = {
    "dataset": DATASET_ID, "revision": dataset_sha,
    "mode": INFERENCE_MODE, "embedding_model": EMBED_MODEL,
    "page_elements_url": PAGE_ELEMENTS_URL if INFERENCE_MODE == "hosted" else None,
    "ocr_url": OCR_URL if INFERENCE_MODE == "hosted" else None,
    "embed_url": EMBED_URL if INFERENCE_MODE == "hosted" else None,
    "corpus_format": "single-page-image-pdf-150dpi-v1",
}
config_hash = hashlib.sha256(json.dumps(index_config, sort_keys=True).encode()).hexdigest()[:12]
lancedb_uri = INDEX_DIR / f"{INFERENCE_MODE}-{config_hash}"
lancedb_uri.mkdir(parents=True, exist_ok=True)
table_name = "vidore_v3_hr"
manifest_path = lancedb_uri / "notebook-index.json"
database = lancedb.connect(str(lancedb_uri))
table_exists = table_name in database.list_tables(limit=None).tables
if table_exists and not OVERWRITE_INDEX:
    assert manifest_path.is_file(), "Index has no completed-ingestion manifest. Set OVERWRITE_INDEX=True."
    assert json.loads(manifest_path.read_text()) == index_config, "Index configuration differs. Set OVERWRITE_INDEX=True."
    print(f"Reusing {lancedb_uri}/{table_name}")
else:
    # Remove the completion marker before rebuilding so failed runs cannot be reused.
    manifest_path.unlink(missing_ok=True)
    command = [
        retriever_cli, "ingest", "batch", str(corpus_dir),
        "--lancedb-uri", str(lancedb_uri), "--table-name", table_name,
        "--embed-model-name", EMBED_MODEL,
    ]
    if INFERENCE_MODE == "local":
        command += ["--local-ingest-embed-backend", "hf"]
    else:
        command += ["--page-elements-invoke-url", PAGE_ELEMENTS_URL,
                    "--ocr-invoke-url", OCR_URL, "--embed-invoke-url", EMBED_URL]
    if table_exists:
        command.append("--overwrite")
    subprocess.run(command, check=True)
    database = lancedb.connect(str(lancedb_uri))
    assert database.open_table(table_name).count_rows() > 0, "Ingestion produced an empty table."
    manifest_path.write_text(json.dumps(index_config, indent=2) + "\n")
row_count = database.open_table(table_name).count_rows()
assert row_count > 0, "The retrieval table is empty."
print(f"Indexed elements: {row_count}")


## 6. Retrieve and rank unique pages

Embed queries with the same model used for ingestion. Extracted elements are ranked
by vector similarity; keep each corpus page's first occurrence. Fetch more candidates
when duplicates leave fewer than ten unique pages, up to `MAX_CANDIDATES` or the table size.
Unknown or missing source identifiers fail validation rather than silently lowering scores.


In [ ]:
embed_kwargs = {"model_name": EMBED_MODEL, "embed_model_name": EMBED_MODEL}
if INFERENCE_MODE == "local":
    embed_kwargs["local_ingest_embed_backend"] = "hf"
else:
    embed_kwargs.update(embedding_endpoint=EMBED_URL, api_key=os.environ.get("NVIDIA_API_KEY", ""))
retriever = Retriever(
    run_mode="local" if INFERENCE_MODE == "local" else "service",
    vdb_kwargs={"uri": str(lancedb_uri), "table_name": table_name},
    embed_kwargs=embed_kwargs, top_k=INITIAL_CANDIDATES,
)

def page_run(ids, hits):
    result = build_beir_run_from_hits(ids, hits, doc_id_field="pdf_basename")
    for qid, query_hits in zip(ids, hits):
        for hit in query_hits:
            single = build_beir_run_from_hits([qid], [[hit]], doc_id_field="pdf_basename")[qid]
            assert single and set(single) <= corpus_ids, f"Unrecognized corpus page metadata for query {qid}."
    return result

run = {}
retrieval_depths = {}
candidate_limit = min(MAX_CANDIDATES, row_count)
target_pages = min(max(CUTOFFS), len(corpus_ids))
for start in tqdm(range(0, len(queries), QUERY_BATCH_SIZE), desc="Retrieving query batches"):
    ids = query_ids[start:start + QUERY_BATCH_SIZE]
    texts = dict(zip(ids, queries[start:start + QUERY_BATCH_SIZE]))
    pending = ids
    depth = min(INITIAL_CANDIDATES, candidate_limit)
    while pending:
        hits = retriever.queries([texts[qid] for qid in pending], top_k=depth)
        assert len(hits) == len(pending), "Retriever returned an unexpected number of results."
        batch_run = page_run(pending, hits)
        run.update(batch_run)
        retrieval_depths.update({qid: depth for qid in pending})
        pending = [qid for qid in pending if len(run[qid]) < target_pages]
        if depth >= candidate_limit:
            break
        depth = min(depth * 2, candidate_limit)
insufficient = sum(len(run[qid]) < target_pages for qid in query_ids)
if insufficient:
    print(f"{insufficient} queries returned fewer than {target_pages} unique pages at the candidate limit "
          f"({candidate_limit}); empty or short rankings are still scored. Increase MAX_CANDIDATES if needed.")


## 7. Report Recall and nDCG

Recall@k is the fraction of all positively labeled pages found in the top k unique pages.
nDCG@k rewards relevant pages appearing early, using the qrels' relevance grades and
logarithmic rank discount (the repository's BEIR metric implementation).
Both metrics are averaged over every selected query, including empty retrieval results.

The summary has one row per cutoff. Inspect `per_query_df` for individual rankings and
scores. CSV reports, the ranked run, and the run configuration are saved for comparison;
credentials are excluded.


In [ ]:
metrics = compute_beir_metrics(qrels, run, ks=CUTOFFS)
summary_df = pd.DataFrame(
    [{"k": k, "Recall": metrics[f"recall@{k}"], "nDCG": metrics[f"ndcg@{k}"]} for k in CUTOFFS]
).set_index("k")
rows = []
for qid, query in zip(query_ids, queries):
    rows.append({
        "query_id": qid, "query": query,
        "relevant_pages": [doc_id for doc_id, score in qrels[qid].items() if score > 0],
        "ranked_pages": list(run[qid])[:max(CUTOFFS)],
        "unique_pages_retrieved": len(run[qid]),
        "candidates_requested": retrieval_depths[qid],
        **compute_beir_metrics({qid: qrels[qid]}, {qid: run[qid]}, ks=CUTOFFS),
    })
per_query_df = pd.DataFrame(rows)
print(f"Page-level retrieval metrics over {len(query_ids)} queries ({INFERENCE_MODE}):")
display(summary_df.style.format("{:.4f}"))
display(per_query_df.head())

from datetime import datetime, timezone
output_dir = RESULTS_DIR / f"{INFERENCE_MODE}-{config_hash}-{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}"
output_dir.mkdir(parents=True, exist_ok=False)
summary_df.to_csv(output_dir / "summary.csv")
per_query_df.to_csv(output_dir / "per_query.csv", index=False)
(output_dir / "run.json").write_text(json.dumps(run, indent=2) + "\n")
(output_dir / "config.json").write_text(json.dumps({
    **index_config, "max_queries": MAX_QUERIES, "evaluated_queries": len(query_ids),
    "cutoffs": CUTOFFS, "query_batch_size": QUERY_BATCH_SIZE,
    "initial_candidates": INITIAL_CANDIDATES, "max_candidates": MAX_CANDIDATES,
    "queries_below_target_pages": insufficient,
}, indent=2) + "\n")
print(f"Saved reports to {output_dir}")
